In [1]:
# A. Membangun Katalog (Create)
from pymongo import MongoClient

# Inisialisasi koneksi ke MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Pembersihan data awal agar pengujian mulai dari kondisi bersih
koleksi_menu.delete_many({})

# Menyisipkan 12 dokumen menu ke dalam collection "menu"
koleksi_menu.insert_many([
    {"nama_menu": "Espresso", "kategori": "Kopi", "harga": 18000, "tingkat_kepahitan": 5, "stok": 20},
    {"nama_menu": "Americano", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 4, "stok": 25},
    {"nama_menu": "Kopi Tubruk", "kategori": "Kopi", "harga": 15000, "tingkat_kepahitan": 5, "rekomendasi_barista": True},
    {"nama_menu": "Latte", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 2, "stok": 30},
    {"nama_menu": "Cappuccino", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 3, "stok": 15},
    {"nama_menu": "Caramel Macchiato", "kategori": "Kopi", "harga": 32000, "tingkat_kepahitan": 2, "stok": 10},
    {"nama_menu": "Nasi Goreng Ayam", "kategori": "Makanan", "harga": 25000, "pedas_level": 2, "stok": 40},
    {"nama_menu": "Mie Goreng Jawa", "kategori": "Makanan", "harga": 22000, "pedas_level": 1, "stok": 35},
    {"nama_menu": "Ayam Geprek", "kategori": "Makanan", "harga": 27000, "pedas_level": 3, "rekomendasi_barista": True},
    {"nama_menu": "Roti Bakar Cokelat Keju", "kategori": "Makanan", "harga": 18000, "pedas_level": 0, "stok": 50},
    {"nama_menu": "Kentang Goreng Original", "kategori": "Makanan", "harga": 16000, "pedas_level": 0, "stok": 45},
    {"nama_menu": "Cireng Bumbu Rujak", "kategori": "Makanan", "harga": 14000, "pedas_level": 1, "stok": 30}
])

print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Jumlah dokumen di collection 'menu': 12


In [3]:
# B. Kueri Bertingkat (Read)

# 1. Seluruh menu kategori "Kopi" dengan tingkat_kepahitan > 3
print("1. Menu Kopi dengan tingkat kepahitan > 3:")
for p in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {p['nama_menu']} (Kepahitan: {p['tingkat_kepahitan']})")

print("\n")

# 2. Seluruh menu dengan harga antara 15000 dan 30000
print("2. Menu dengan harga antara Rp15.000 - Rp30.000:")
for p in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {p['nama_menu']} (Rp{p['harga']:,})")

print("\n")

# 3. Menu yang tidak memiliki field "pedas_level" sama sekali
print("3. Menu yang tidak memiliki field 'pedas_level':")
for p in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {p['nama_menu']} ({p['kategori']})")

1. Menu Kopi dengan tingkat kepahitan > 3:
  - Espresso (Kepahitan: 5)
  - Americano (Kepahitan: 4)
  - Kopi Tubruk (Kepahitan: 5)


2. Menu dengan harga antara Rp15.000 - Rp30.000:
  - Espresso (Rp18,000)
  - Americano (Rp22,000)
  - Kopi Tubruk (Rp15,000)
  - Latte (Rp28,000)
  - Cappuccino (Rp28,000)
  - Nasi Goreng Ayam (Rp25,000)
  - Mie Goreng Jawa (Rp22,000)
  - Ayam Geprek (Rp27,000)
  - Roti Bakar Cokelat Keju (Rp18,000)
  - Kentang Goreng Original (Rp16,000)


3. Menu yang tidak memiliki field 'pedas_level':
  - Espresso (Kopi)
  - Americano (Kopi)
  - Kopi Tubruk (Kopi)
  - Latte (Kopi)
  - Cappuccino (Kopi)
  - Caramel Macchiato (Kopi)


In [4]:
# C. Pembaruan Data (Update)

# Tambahkan field status = "Tersedia" ke seluruh dokumen
koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

# Ubah status menjadi "Habis" khusus untuk 2 menu pilihan
koleksi_menu.update_one(
    {"nama_menu": "Americano"},
    {"$set": {"status": "Habis"}}
)

koleksi_menu.update_one(
    {"nama_menu": "Ayam Geprek"},
    {"$set": {"status": "Habis"}}
)

print("Daftar status menu setelah update:")
for p in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "kategori": 1, "status": 1}):
    print(f"  - {p['nama_menu']} ({p['kategori']}): {p['status']}")

Daftar status menu setelah update:
  - Espresso (Kopi): Tersedia
  - Americano (Kopi): Habis
  - Kopi Tubruk (Kopi): Tersedia
  - Latte (Kopi): Tersedia
  - Cappuccino (Kopi): Tersedia
  - Caramel Macchiato (Kopi): Tersedia
  - Nasi Goreng Ayam (Makanan): Tersedia
  - Mie Goreng Jawa (Makanan): Tersedia
  - Ayam Geprek (Makanan): Habis
  - Roti Bakar Cokelat Keju (Makanan): Tersedia
  - Kentang Goreng Original (Makanan): Tersedia
  - Cireng Bumbu Rujak (Makanan): Tersedia


In [8]:
# D. Integrasi ke Spark

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantaraSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 2. Ambil seluruh data dari MongoDB
dokumen_mongo = list(koleksi_menu.find({}))

# 3. Ubah ke Pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# 4. Penyesuaian Tipe Data (Mencegah Error CANNOT_MERGE_TYPE)
# - Ubah ObjectId MongoDB menjadi String
df_pandas["_id"] = df_pandas["_id"].astype(str)

# - Tangani kolom boolean opsional agar tidak bercampur dengan NaN (float)
if "rekomendasi_barista" in df_pandas.columns:
    df_pandas["rekomendasi_barista"] = df_pandas["rekomendasi_barista"].fillna(False)

# - Ubah seluruh NaN Pandas menjadi None agar PySpark membacanya sebagai null
df_pandas = df_pandas.astype(object).where(pd.notnull(df_pandas), None)

# 5. Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# 6. Tampilkan skema DataFrame
print("--- Skema Spark DataFrame ---")
df_spark_menu.printSchema()

# 7. Rata-rata harga per kategori
print("--- Rata-rata Harga per Kategori ---")
df_spark_menu.groupBy("kategori").agg(
    round(avg("harga"), 2).alias("rata_rata_harga")
).show()

/tmp/ipykernel_31992/1935517770.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pandas["rekomendasi_barista"] = df_pandas["rekomendasi_barista"].fillna(False)


--- Skema Spark DataFrame ---
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: double (nullable = true)
 |-- status: string (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- pedas_level: double (nullable = true)

--- Rata-rata Harga per Kategori ---
+--------+---------------+
|kategori|rata_rata_harga|
+--------+---------------+
|    Kopi|       23833.33|
| Makanan|       20333.33|
+--------+---------------+



# E. Kesimpulan

Tantangan utama saat mengonversi data dari MongoDB ke Spark DataFrame adalah perbedaan karakteristik skema antara kedua sistem, yang memicu error ketidakcocokan tipe data. MongoDB menerapkan flexible schema di mana dokumen bervariasi (contoh: field rekomendasi_barista atau pedas_level hanya ada di sebagian dokumen). Saat dimuat ke Pandas, field yang tidak ada tersebut diisi dengan nilai NaN (bertipe float). Ketika dikonversi ke PySpark, muncul error CANNOT_MERGE_TYPE karena Spark menolak pencampuran tipe data (misalnya Boolean dengan NaN/Float) dalam satu kolom.
Untuk mengatasi tantangan ini, saya melakukan pra-pemrosesan data menggunakan Pandas. Pertama, nilai kosong pada kolom spesifik diatasi dengan fungsi .fillna(). Kedua, seluruh sisa nilai NaN diubah secara eksplisit menjadi None menggunakan perintah .where(pd.notnull()) agar PySpark dapat menerjemahkannya sebagai nilai null yang aman. Selain itu, tipe data BSON ObjectId juga dikonversi menjadi string. Melalui pembersihan ini, skema data berhasil diseragamkan dan analisis PySpark dapat berjalan tanpa error.